In [1]:
from pathlib import Path
TRITON_API_KEY = Path('../api-key.txt').expanduser().read_text(encoding='utf-8').splitlines()[0].strip()
BASE_URL = 'https://tritonai-api.ucsd.edu'

In [2]:
from typing import Dict, Any
from typing import List as listtype
import re

In [3]:
# document loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document

# text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# retriever
from langchain_core.retrievers import BaseRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import TFIDFRetriever, AmazonKnowledgeBasesRetriever

# cross encoder for reranking
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

In [4]:
# rapidfire imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

[https://oss-docs.rapidfire.ai/en/latest/ragspecs.html](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

In [5]:
class TextLoaderWithMetadata(TextLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as TextLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using TextLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [6]:
def metadata_func(document, metadata):
    # extract the page title from the content and store it in metadata
    title = document.page_content.split('\n')[0].split('===')[0]
    metadata['title'] = title
    metadata['source'] = Path(metadata['source']).name
    return metadata

In [7]:
loader = DirectoryLoader(
    path='../data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=TextLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
    },
    show_progress=True,
    use_multithreading=True
)

In [8]:
docs = loader.load()

100%|██████████| 31/31 [00:00<00:00, 5394.94it/s]


In [9]:
docs[0].metadata

{'source': 'fitfunctions.rst',
 'title': 'API: User-Provided Functions for Run Fit'}

Models:
- `gpt2` / `r50k_base`: older ~50k-vocab GPT-2 style tokenizers. `r50k_base` is the standard GPT-2 BPE; `gpt2` is the GPT-2 encoding entry in `tiktoken`. Both use the same regex pattern and the same end-of-text token, with `gpt2` built from the original GPT-2 encoder/vocab files
- `p50k_base`: also ~50k vocab, but different merge ranks from `r50k_base`; intended for Codex-style/code-heavy tokenization
- `p50k_edit`: same base merges as `p50k_base`, but adds extra FIM/edit special tokens like `<|fim_prefix|>`, `<|fim_middle|>`, and `<|fim_suffix|>`
- `cl100k_base`: newer ~100k-vocab tokenizer with a different regex and a larger special-token set; commonly associated with GPT-4 / GPT-3.5-era model families
- `o200k_base`: newer ~200k-vocab tokenizer with a different regex again; used by newer model families such as `gpt-4o`
- `o200k_harmony`: derived from `o200k_base` but with many additional special tokens layered on top

In [10]:
# text splitter 
text_splitter = RecursiveCharacterTextSplitter().from_tiktoken_encoder(
     encoding_name='o200k_base',
     chunk_size=512, 
     chunk_overlap=64,
)
text_splitter._add_start_index = True

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`embedding_cfg (dict[str, Any], optional)`

The embedding class and its kwargs to convert a chunk/query into a vector, provided as a single dictionary. Must include a key `"class"` with the class itself as value, not an instance. Options for the class include `HuggingFaceEmbeddings` and `OpenAIEmbeddings`. 

The kwargs that follow must contain all parameters needed to initialize the embedding class; required parameters vary by embedding class. For example, `HuggingFaceEmbeddings` needs `model_name`, `model_kwargs` and `device`, while `OpenAIEmbeddings` needs `"model"` and `"api_key"`.

In [11]:
batch_size = 32

embedding_cfg = List([
    # yet to use open ai embeddings
    {
        'class': HuggingFaceEmbeddings,
        'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
        'model_kwargs': {'device': 'cpu'},
        'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size}
    },
    # {
    #     'class': OpenAIEmbeddings,
    #     'model': 'api-tgpt-embeddings',
    #     'api_key': TRITON_API_KEY,
    #     'base_url': BASE_URL,
    #     'check_embedding_ctx_length': False
    # }
])

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`vector_store_cfg (dict[str, Any], optional)`

The vector store type and args to store and possibly index embedding vectors for retrieval, provided as a single dictionary.

- `"type"`: The type of vector store to use. Must be one of `"faiss"`, `"pgvector"`, or `"pinecone"`. Required.
- `"batch_size"`: Number of vectors per insert batch. Applies to all 3 types of stores. Optional; default is 128.

In [12]:
vector_store_cfg={
    'type': 'faiss', 
    'batch_size': batch_size
}

In [13]:
search_cfg=List([
    {'type': 'similarity', 'k': 10},
    {'type': 'mmr', 'k': 10}
])

In [14]:
reranker_cfg={
    'class': CrossEncoderReranker,
    'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
    'model_kwargs': {'device': 'cpu'},
    'top_n': 5,
}

In [15]:
def custom_template(doc):
    return f'{doc.metadata["title"]}: {doc.page_content}'

In [16]:
rag = RFLangChainRagSpec(
    # to load the corpus
    document_loader=loader,
    # to split documents into chunks
    text_splitter=text_splitter,
    # to embed the chunks
    embedding_cfg=embedding_cfg,
    # vectore storage
    vector_store_cfg=vector_store_cfg,
    # search algorithm
    search_cfg=search_cfg,
    # rerank top results with a cross encoder
    reranker_cfg=reranker_cfg,
    # whether to use gpu-accelerated faiss
    enable_gpu_search=False,
    # to format each retrieved chunk for context injection into prompts
    document_template=custom_template,
)

In [17]:
INSTRUCTIONS = """
You are a helpful assistant answering questions about RapidFire AI docuentation.

Your job:
- Read the retrieved documentation context
- Answer the user's question using only the context
- Do not make up information that is not supported by the retrieved text

Rules:
1. Use only the retrieved document context.
2. Keep the answer concise but complete.
3. Mention relevant file names when they are clearly useful.

Output format:
#### ANSWER: <final answer>
"""

In [18]:
import json
from datasets import Dataset

In [19]:
def load_eval_question_set(path, encoding='utf-8', validation=False):
    # load the questions from the file
    with open(path, 'r', encoding=encoding) as f:
        eval_questions = json.load(f)
    
    # to store each example
    eval_rows = []

    for example in eval_questions:
        # extract the question and id
        row = {
            'question_id': example['question_id'],
            'question': example['question']
        }
        # validation sets include reference answer an source evidence
        if validation:
            row['reference_answer'] = example['reference_answer']
            row['source_evidence'] = example['source_evidence']
        
        eval_rows.append(row)

    
    # convert into a hugging face dataset
    eval_dataset = Dataset.from_list(eval_rows)
    return eval_dataset

In [20]:
golden_qa_val = load_eval_question_set(path='../data/validation-set/validation-set-golden-qa-pairs.json', validation=True)
print(golden_qa_val)
print(golden_qa_val[1].keys())
print(golden_qa_val[1])

Dataset({
    features: ['question_id', 'question', 'reference_answer', 'source_evidence'],
    num_rows: 45
})
dict_keys(['question_id', 'question', 'reference_answer', 'source_evidence'])
{'question_id': 2, 'question': "How does RapidFire AI's adaptive execution engine differ from traditional sequential execution for multi-config experiments?", 'reference_answer': "Traditional tools force you to run one config after another sequentially. In contrast, RapidFire AI's adaptive execution engine shards the dataset randomly into chunks and schedules all runs on one shard at a time, then cycles through all shards. This enables interruptible execution across GPUs/CPUs, providing a far more concurrent learning experience by automatically swapping adapters and base models across GPUs and DRAM via efficient shared memory-based caching mechanisms. The swapping overhead is minimal, less than 5% of runtime. This means before the third model config even starts in the sequential approach, RapidFire 

In [21]:
def sample_preprocess_fn(batch: Dict[str, listtype], 
                         rag: RFLangChainRagSpec, 
                         prompt_manager: RFPromptManager) -> Dict[str, listtype]:
    '''
    convert a batch of dataset rows into model-ready prompts
    '''
    # retrieve context for each question in the batch
    all_context = rag.get_context(batch_queries=batch['question'], serialize=False)
    # extract the source metadata field from each retrieved document
    source_evidence = []
    for docs in all_context:
        evidence_for_query = []
        for doc in docs:
            evidence_for_query.append({
                'source': doc.metadata.get('source'),
                'start_index': doc.metadata.get('start_index')
            })
        source_evidence.append(evidence_for_query)
    # convert retrieved docs into plain text strings suitable for prompt injection
    serialized_context = rag.serialize_documents(all_context)

    # build a chat-style prompt per question
    prompts = [
        [
            # instruction as system prompt
            {'role': 'system', 'content': INSTRUCTIONS},
            # user message containing the question
            {
                'role': 'user', 'content': f'Question: {question} \nDocumentation context: {context} \nYour response:'
            }
        ]
        for question, context in zip(batch['question'], serialized_context)
    ]

    return {
        'prompts': prompts,
        'source_evidence': source_evidence,
        **batch
    }

In [22]:
def extract_solution(answer: str) -> str:
    '''
    extract the answer
    '''
    # search for the answer marker
    match = re.search(
        r"####\s*ANSWER:\s*(.*)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    # if the marker is missing fall back to returning the whole stripped text
    return answer.strip()

In [23]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    '''
    turn raw generated text into a field
    '''
    batch['answer'] = [extract_solution(answer) for answer in batch['generated_text']]
    return batch

In [24]:
openai_config = RFOpenAIAPIModelConfig(
    client_config={'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2},
    model_config={
        'model': 'api-mistral-small-3.2-2506',
        'max_completion_tokens': 2048,
        # two different reasoning levels
        # 'reasoning_effort': List(['medium', 'high'])
    },
    # request per minute needs to be set based on your account tier and the specific model used
    rpm_limit=120,
    # token per minute needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000,
    # rag model
    rag=rag,
    prompt_manager=None
)

In [25]:
# skip metrics for now
def compute_metrics_fn(batch):
    return {}

def accumulate_metrics_fn(all_metrics):
    return {}

In [26]:
config_set = List([{
    'openai_config': List([
        openai_config
    ]),
    'batch_size': batch_size,
    'preprocess_fn': sample_preprocess_fn,
    'postprocess_fn': sample_postprocess_fn,
    # add metrics later
    'compute_metrics_fn': compute_metrics_fn,
    'accumulate_metrics_fn': accumulate_metrics_fn,
    # 'online_strategy_kwargs': {
    #     'strategy_name': 'normal',
    #     'confidence_level': 0.95,
    #     'use_fpc': True,
    # }
}])

config_group = RFGridSearch(config_set)

In [27]:
# create experiment
experiment = Experiment(
    experiment_name='rag-generation-only',
    mode='evals'
)

# run generation
results = experiment.run_evals(
    config_group=config_group,
    dataset=golden_qa_val,
    num_shards=4,
    seed=42
)
experiment.end()

An experiment with the same name already exists. Created a new experiment 'rag-generation-only_6' with Experiment ID: 7 at /Users/chenchiahan/rapidfireai/rapidfire_experiments/rag-generation-only_6
Created directory: /Users/chenchiahan/rapidfireai/logs/rag-generation-only_6


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,13.3s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.class,embedding_cfg.encode_kwargs.batch_size,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model_name,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Processing Time,Samples Per Second,Samples Processed,Throughput
1,api-mistral-small-3.2-2506,COMPLETED,4/4,-,RecursiveCharacterTextSplitter,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-MiniLM-L6-v2,32,faiss,-,10,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=2048,80.76 seconds,0.56,45,0.7/s
2,api-mistral-small-3.2-2506,COMPLETED,4/4,-,RecursiveCharacterTextSplitter,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-MiniLM-L6-v2,32,faiss,20,10,0.5,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,max_completion_tokens=2048,64.38 seconds,0.70,45,0.7/s


Experiment rag-generation-only_6 ended


In [29]:
# inspect results
print(type(results))
print(results[1][0].keys())

<class 'dict'>
dict_keys(['prompts', 'source_evidence', 'question_id', 'question', 'reference_answer', 'generated_text', 'answer'])


In [ ]:
print('-' * 30)
print('Question: ')
print(results[1][0]['question'])
print('-' * 30)
print('Reference Answer: ')
print(results[1][0]['reference_answer'])
print('-' * 30)
print('Generated Answer: ')
print(results[1][0]['answer'])
print('-' * 30)
print('Source Evidence: ')
print(results[1][0]['source_evidence'])

------------------------------
Question: 
['What are the two knob set generators currently supported by RapidFire AI for creating multi-config specifications?', "How does RapidFire AI's adaptive execution engine differ from traditional sequential execution for multi-config experiments?", 'What parameters does the Experiment constructor accept?', 'What is the difference between RFGridSearch and RFRandomSearch in terms of how they handle knob values?', 'How do I set up RapidFire AI for RAG evaluation on a machine without GPUs?', 'What are the four Interactive Control (IC) Operations supported by RapidFire AI?', 'What does the preprocess_fn function do in the run_evals pipeline, and what arguments does the system inject into it?', 'What three vector store backends does RapidFire AI support and what modes of operation do they offer?', 'What columns does the DataFrame returned by get_runs_info() contain?', 'What are the three search algorithm types supported in the search_cfg of RFLangChain

: 

In [ ]:
print(results[1][0]['source_evidence'])